In [1]:
# 1. Imports

from pathlib import Path
import time
import configparser

import ccxt
import pandas as pd
import requests

In [ ]:
# 2. Configuration

RUN_DATE = "2026-03-13"
TOP_N = 100
VS_CURRENCY = "usd"

MIN_GLOBAL_VOLUME_24H_USD = 50_000_000
MIN_MAJOR_EXCHANGE_COUNT = 10
TOP_MAJOR_EXCHANGES_N = 20

PROCESSED_DIR = Path("../../data/processed/raw")
ASSETS_PATH = PROCESSED_DIR / "assets.parquet"
ASSET_DAILY_PATH = PROCESSED_DIR / "asset_daily.parquet"

COINGECKO_BASE_URL = "https://api.coingecko.com/api/v3"
REQUEST_SLEEP_SECONDS = 2.1
REQUEST_TIMEOUT_SECONDS = 30
USER_AGENT = "BachelorarbeitDataPipeline/1.0"

TARGET_VOLUME_EXCHANGE = "binance"
USD_LIKE_QUOTES = ["USDT", "USDC", "USD"]

config = configparser.ConfigParser()
config.read("../../secret.cfg")

COINGECKO_API_KEY = config.get("COINGECKO", "DEMO_API_KEY")

if not COINGECKO_API_KEY:
    raise ValueError("CoinGecko DEMO_API_KEY not found in secret.cfg")

ASSET_COLUMNS = [
    "asset_id",
    "coingecko_id",
    "symbol",
    "name",
    "circulating_supply",
    "total_supply",
    "max_supply",
]

ASSET_DAILY_COLUMNS = [
    "date",
    "asset_id",
    "price_usd",
    "market_cap_usd",
    "global_volume_24h_usd",
    "binance_volume_24h_usd",
]

In [ ]:
# 3. Request helper

def coingecko_get(endpoint: str, params: dict | None = None, max_retries: int = 5) -> dict | list:
    url = f"{COINGECKO_BASE_URL}{endpoint}"
    headers = {
        "User-Agent": USER_AGENT,
        "x-cg-demo-api-key": COINGECKO_API_KEY,
    }
    params = dict(params or {})
    last_error = None

    for attempt in range(max_retries):
        response = requests.get(url, params=params, headers=headers, timeout=REQUEST_TIMEOUT_SECONDS)

        if response.status_code == 429:
            wait_seconds = min(90, 5 * (attempt + 1))
            print(f"429 for {endpoint} | attempt {attempt + 1}/{max_retries} | sleeping {wait_seconds}s")
            time.sleep(wait_seconds)
            last_error = RuntimeError(f"429 rate limit for {url}")
            continue

        response.raise_for_status()
        time.sleep(REQUEST_SLEEP_SECONDS)
        return response.json()

    raise last_error or RuntimeError(f"Request failed for {url}")

In [4]:
# API-test
# print(coingecko_get("/ping"))

In [5]:
# 4. Helper functions

def ensure_output_dir() -> None:
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


def enforce_columns(df: pd.DataFrame, expected_columns: list[str]) -> pd.DataFrame:
    missing = [c for c in expected_columns if c not in df.columns]
    extra = [c for c in df.columns if c not in expected_columns]

    if missing:
        raise ValueError(f"Missing columns: {missing}")
    if extra:
        raise ValueError(f"Unexpected columns: {extra}")

    return df[expected_columns].copy()


def make_binance_exchange():
    exchange = ccxt.binance(
        {
            "enableRateLimit": True,
            "timeout": REQUEST_TIMEOUT_SECONDS * 1000,
        }
    )
    exchange.load_markets()
    return exchange


def find_binance_usd_like_symbol(exchange, base_symbol: str) -> str | None:
    base_symbol = str(base_symbol).upper().strip()

    for quote in USD_LIKE_QUOTES:
        candidate = f"{base_symbol}/{quote}"
        if candidate in exchange.markets:
            return candidate

    return None


def fetch_binance_quote_volume_for_date(exchange, market_symbol: str, run_date: str) -> float | None:
    market = exchange.market(market_symbol)

    start_ts = exchange.parse8601(f"{run_date}T00:00:00Z")
    end_ts = start_ts + 24 * 60 * 60 * 1000

    rows = exchange.publicGetKlines(
        {
            "symbol": market["id"],
            "interval": "1d",
            "startTime": start_ts,
            "endTime": end_ts,
            "limit": 1,
        }
    )

    if not rows:
        return None

    row = rows[0]

    candle_date = pd.to_datetime(int(row[0]), unit="ms", utc=True).strftime("%Y-%m-%d")
    if candle_date != run_date:
        return None

    return float(row[7])

In [ ]:
# 5. Fetch top-N universe by market cap

markets_data = coingecko_get(
    "/coins/markets",
    params={
        "vs_currency": VS_CURRENCY,
        "order": "market_cap_desc",
        "per_page": TOP_N,
        "page": 1,
        "sparkline": "false",
        "price_change_percentage": "24h",
    },
)

top_assets_df = pd.DataFrame(markets_data)

required_market_cols = [
    "id",
    "symbol",
    "name",
    "current_price",
    "market_cap",
    "total_volume",
    "circulating_supply",
    "total_supply",
    "max_supply",
]

missing_market_cols = [c for c in required_market_cols if c not in top_assets_df.columns]
if missing_market_cols:
    raise ValueError(f"CoinGecko response missing required fields: {missing_market_cols}")

top_assets_df = top_assets_df[required_market_cols].copy()
top_assets_df

,id,symbol,name,current_price,market_cap,total_volume,circulating_supply,total_supply,max_supply
0,bitcoin,btc,Bitcoin,73342.000000,1468078846334,5.181936e+10,2.000281e+07,2.000281e+07,2.100000e+07
1,ethereum,eth,Ethereum,2279.590000,275379358516,3.396717e+10,1.206919e+08,1.206919e+08,NaN
2,tether,usdt,Tether,0.999985,184031727331,9.525152e+10,1.840323e+11,1.894993e+11,NaN
3,binancecoin,bnb,BNB,672.520000,91790040092,1.614401e+09,1.363579e+08,1.363579e+08,2.000000e+08
4,ripple,xrp,XRP,1.500000,91571365386,3.650243e+09,6.122783e+10,9.998570e+10,1.000000e+11
...,...,...,...,...,...,...,...,...,...
95,fetch-ai,fet,Artificial Superintelligence Alliance,0.234631,524098816,2.327987e+08,2.260137e+09,2.714385e+09,2.714385e+09
96,virtual-protocol,virtual,Virtuals Protocol,0.774129,508538205,1.044817e+08,6.563020e+08,1.000000e+09,1.000000e+09
97,true-usd,tusd,TrueUSD,0.999812,494318107,8.101426e+06,4.945151e+08,4.945151e+08,NaN
98,pancakeswap-token,cake,PancakeSwap,1.490000,491644923,6.511604e+07,3.294786e+08,3.427484e+08,4.000000e+08


In [ ]:
# 6. Build asset table

asset_df = top_assets_df[["id", "symbol", "name", "circulating_supply", "total_supply", "max_supply"]].copy()

asset_df = asset_df.rename(
    columns={
        "id": "coingecko_id",
    }
)

asset_df["symbol"] = asset_df["symbol"].str.upper()
asset_df.insert(0, "asset_id", range(1, len(asset_df) + 1))

asset_df = enforce_columns(asset_df, ASSET_COLUMNS)
asset_df

,asset_id,coingecko_id,symbol,name,circulating_supply,total_supply,max_supply
0,1,bitcoin,BTC,Bitcoin,2.000281e+07,2.000281e+07,2.100000e+07
1,2,ethereum,ETH,Ethereum,1.206919e+08,1.206919e+08,NaN
2,3,tether,USDT,Tether,1.840323e+11,1.894993e+11,NaN
3,4,binancecoin,BNB,BNB,1.363579e+08,1.363579e+08,2.000000e+08
4,5,ripple,XRP,XRP,6.122783e+10,9.998570e+10,1.000000e+11
...,...,...,...,...,...,...,...
95,96,fetch-ai,FET,Artificial Superintelligence Alliance,2.260137e+09,2.714385e+09,2.714385e+09
96,97,virtual-protocol,VIRTUAL,Virtuals Protocol,6.563020e+08,1.000000e+09,1.000000e+09
97,98,true-usd,TUSD,TrueUSD,4.945151e+08,4.945151e+08,NaN
98,99,pancakeswap-token,CAKE,PancakeSwap,3.294786e+08,3.427484e+08,4.000000e+08


In [ ]:
# 7. Fetch top major exchanges globally

exchanges_data = coingecko_get(
    "/exchanges",
    params={
        "per_page": 100,
        "page": 1,
    },
)

exchanges_df = pd.DataFrame(exchanges_data)

required_exchange_cols = [
    "id",
    "name",
    "trust_score_rank",
    "trade_volume_24h_btc",
]

missing_exchange_cols = [c for c in required_exchange_cols if c not in exchanges_df.columns]
if missing_exchange_cols:
    raise ValueError(f"CoinGecko exchange response missing required fields: {missing_exchange_cols}")

exchanges_df = exchanges_df[required_exchange_cols].copy()

major_exchanges_df = (
    exchanges_df
    .sort_values(["trade_volume_24h_btc", "trust_score_rank"], ascending=[False, True])
    .head(TOP_MAJOR_EXCHANGES_N)
    .reset_index(drop=True)
)

major_exchange_ids = set(major_exchanges_df["id"].tolist())

major_exchanges_df

,id,name,trust_score_rank,trade_volume_24h_btc
0,binance,Binance,1,166992.494269
1,coinup,CoinUp.io,92,107342.974406
2,pionex,Pionex,54,65195.542865
3,biconomy,Biconomy.com,55,58028.361696
4,coinw,CoinW,22,56326.007756
5,mxc,MEXC,8,52471.985112
6,coinstore,Coinstore,9,50440.867163
7,lbank,LBank,10,47422.943266
8,tapbit,Tapbit,23,46801.450494
9,bitmart,BitMart,24,46043.113954


In [ ]:
# 8. Compute major exchange count per asset

major_exchange_count_rows = []

major_exchange_ids_list = sorted(major_exchange_ids)

for _, asset_row in asset_df.iterrows():
    asset_id = int(asset_row["asset_id"])
    coingecko_id = asset_row["coingecko_id"]

    tickers_data = coingecko_get(
        f"/coins/{coingecko_id}/tickers",
        params={
            "exchange_ids": ",".join(major_exchange_ids_list),
            "page": 1,
        },
    )

    tickers = tickers_data.get("tickers", [])
    matched_exchange_ids = set()

    for ticker in tickers:
        market = ticker.get("market") or {}
        exchange_id = market.get("identifier")

        if exchange_id in major_exchange_ids:
            matched_exchange_ids.add(exchange_id)

    major_exchange_count_rows.append(
        {
            "asset_id": asset_id,
            "major_exchange_count": len(matched_exchange_ids),
        }
    )

major_exchange_count_df = pd.DataFrame(major_exchange_count_rows)
major_exchange_count_df

,asset_id,major_exchange_count
0,1,17
1,2,18
2,3,20
3,4,20
4,5,20
...,...,...
95,96,20
96,97,17
97,98,8
98,99,19


In [10]:
# 9. Build asset_daily table

binance = make_binance_exchange()

binance_volume_rows = []

for _, asset_row in asset_df.iterrows():
    asset_id = int(asset_row["asset_id"])
    symbol = asset_row["symbol"]

    market_symbol = find_binance_usd_like_symbol(binance, symbol)

    volume_value = None
    error_message = None

    if market_symbol is not None:
        try:
            volume_value = fetch_binance_quote_volume_for_date(
                exchange=binance,
                market_symbol=market_symbol,
                run_date=RUN_DATE,
            )
        except Exception as e:
            error_message = repr(e)
            print(f"Volume fetch failed for {symbol} on binance: {e}")

    binance_volume_rows.append(
        {
            "asset_id": asset_id,
            "symbol": symbol,
            "market_symbol": market_symbol,
            "binance_volume_24h_usd": volume_value,
            "error_message": error_message,
        }
    )

binance_debug_df = pd.DataFrame(binance_volume_rows)

asset_daily_df = pd.DataFrame(
    {
        "date": RUN_DATE,
        "asset_id": asset_df["asset_id"],
        "price_usd": top_assets_df["current_price"].values,
        "market_cap_usd": top_assets_df["market_cap"].values,
        "global_volume_24h_usd": top_assets_df["total_volume"].values,
    }
)

asset_daily_df = asset_daily_df.merge(
    binance_debug_df[["asset_id", "binance_volume_24h_usd"]],
    on="asset_id",
    how="left",
)

asset_daily_df["binance_volume_24h_usd"] = pd.to_numeric(
    asset_daily_df["binance_volume_24h_usd"],
    errors="coerce",
)

asset_daily_df = enforce_columns(asset_daily_df, ASSET_DAILY_COLUMNS)
asset_daily_df

,date,asset_id,price_usd,market_cap_usd,global_volume_24h_usd,binance_volume_24h_usd
0,2026-03-13,1,73342.000000,1468078846334,5.181936e+10,2.591769e+09
1,2026-03-13,2,2279.590000,275379358516,3.396717e+10,1.279403e+09
2,2026-03-13,3,0.999985,184031727331,9.525152e+10,1.123857e+07
3,2026-03-13,4,672.520000,91790040092,1.614401e+09,1.539733e+08
4,2026-03-13,5,1.500000,91571365386,3.650243e+09,2.879646e+08
...,...,...,...,...,...,...
95,2026-03-13,96,0.234631,524098816,2.327987e+08,2.375050e+07
96,2026-03-13,97,0.774129,508538205,1.044817e+08,1.579813e+07
97,2026-03-13,98,0.999812,494318107,8.101426e+06,3.336478e+05
98,2026-03-13,99,1.490000,491644923,6.511604e+07,3.375700e+06


In [ ]:
# 10. Build screening table and filter eligible assets

screening_df = (
    asset_df[["asset_id", "coingecko_id", "symbol", "name"]]
    .merge(
        asset_daily_df[["asset_id", "market_cap_usd", "global_volume_24h_usd"]],
        on="asset_id",
        how="left",
    )
    .merge(
        major_exchange_count_df,
        on="asset_id",
        how="left",
    )
)

screening_df["major_exchange_count"] = screening_df["major_exchange_count"].fillna(0).astype(int)
screening_df["market_cap_rank"] = screening_df["market_cap_usd"].rank(method="dense", ascending=False).astype(int)

screening_df["passes_volume_filter"] = screening_df["global_volume_24h_usd"] > MIN_GLOBAL_VOLUME_24H_USD
screening_df["passes_exchange_filter"] = screening_df["major_exchange_count"] >= MIN_MAJOR_EXCHANGE_COUNT
screening_df["passes_market_cap_filter"] = screening_df["market_cap_rank"] <= 100

screening_df["passes_all_filters"] = (
    screening_df["passes_volume_filter"]
    & screening_df["passes_exchange_filter"]
    & screening_df["passes_market_cap_filter"]
)

eligible_assets_df = screening_df.loc[screening_df["passes_all_filters"]].copy()

screening_df.sort_values("market_cap_rank").reset_index(drop=True)

,asset_id,coingecko_id,symbol,name,market_cap_usd,global_volume_24h_usd,major_exchange_count,market_cap_rank,passes_volume_filter,passes_exchange_filter,passes_market_cap_filter,passes_all_filters
0,1,bitcoin,BTC,Bitcoin,1468078846334,5.181936e+10,17,1,True,True,True,True
1,2,ethereum,ETH,Ethereum,275379358516,3.396717e+10,18,2,True,True,True,True
2,3,tether,USDT,Tether,184031727331,9.525152e+10,20,3,True,True,True,True
3,4,binancecoin,BNB,BNB,91790040092,1.614401e+09,20,4,True,True,True,True
4,5,ripple,XRP,XRP,91571365386,3.650243e+09,20,5,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,fetch-ai,FET,Artificial Superintelligence Alliance,524098816,2.327987e+08,20,96,True,True,True,True
96,97,virtual-protocol,VIRTUAL,Virtuals Protocol,508538205,1.044817e+08,17,97,True,True,True,True
97,98,true-usd,TUSD,TrueUSD,494318107,8.101426e+06,8,98,False,False,True,False
98,99,pancakeswap-token,CAKE,PancakeSwap,491644923,6.511604e+07,19,99,True,True,True,True


In [12]:
# 11. Validation

# asset validations
if asset_df["asset_id"].duplicated().any():
    raise ValueError("Duplicate asset_id found in asset_df.")

if asset_df["coingecko_id"].duplicated().any():
    raise ValueError("Duplicate coingecko_id found in asset_df.")

if asset_df["symbol"].isna().any() or (asset_df["symbol"].astype(str).str.strip() == "").any():
    raise ValueError("Empty symbol found in asset_df.")

if asset_df["name"].isna().any() or (asset_df["name"].astype(str).str.strip() == "").any():
    raise ValueError("Empty name found in asset_df.")

# asset_daily validations
if asset_daily_df.duplicated(subset=["date", "asset_id"]).any():
    raise ValueError("Duplicate (date, asset_id) found in asset_daily_df.")

for col in ["market_cap_usd", "global_volume_24h_usd", "binance_volume_24h_usd"]:
    if (asset_daily_df[col].dropna() < 0).any():
        raise ValueError(f"Negative values found in {col}.")

# screening validations
if screening_df["asset_id"].duplicated().any():
    raise ValueError("Duplicate asset_id found in screening_df.")

if (screening_df["major_exchange_count"] < 0).any():
    raise ValueError("Negative major_exchange_count found in screening_df.")

print("All validation checks passed.")

All validation checks passed.


In [ ]:
# 12. Save parquet files


ensure_output_dir()

asset_df.to_parquet(ASSETS_PATH, index=False)
asset_daily_df.to_parquet(ASSET_DAILY_PATH, index=False)

print(f"Saved: {ASSETS_PATH}")
print(f"Saved: {ASSET_DAILY_PATH}")

Saved: ../data/processed/assets.parquet
Saved: ../data/processed/asset_daily.parquet


In [ ]:
# 13. Summary

print("Run summary")
print(f"RUN_DATE: {RUN_DATE}")
print(f"TOP_N: {TOP_N}")
print(f"TOP_MAJOR_EXCHANGES_N: {TOP_MAJOR_EXCHANGES_N}")
print(f"MIN_GLOBAL_VOLUME_24H_USD: {MIN_GLOBAL_VOLUME_24H_USD}")
print(f"MIN_MAJOR_EXCHANGE_COUNT: {MIN_MAJOR_EXCHANGE_COUNT}")
print(f"asset rows: {len(asset_df)}")
print(f"asset_daily rows: {len(asset_daily_df)}")
print(f"eligible asset rows: {len(eligible_assets_df)}")

display(major_exchanges_df)
display(asset_df)
display(asset_daily_df)
display(screening_df.sort_values("market_cap_rank").reset_index(drop=True))
display(eligible_assets_df.sort_values("market_cap_rank").reset_index(drop=True))

Run summary
RUN_DATE: 2026-03-13
TOP_N: 100
TOP_MAJOR_EXCHANGES_N: 20
MIN_GLOBAL_VOLUME_24H_USD: 50000000
MIN_MAJOR_EXCHANGE_COUNT: 10
asset rows: 100
asset_daily rows: 100
eligible asset rows: 52


,id,name,trust_score_rank,trade_volume_24h_btc
0,binance,Binance,1,166992.494269
1,coinup,CoinUp.io,92,107342.974406
2,pionex,Pionex,54,65195.542865
3,biconomy,Biconomy.com,55,58028.361696
4,coinw,CoinW,22,56326.007756
5,mxc,MEXC,8,52471.985112
6,coinstore,Coinstore,9,50440.867163
7,lbank,LBank,10,47422.943266
8,tapbit,Tapbit,23,46801.450494
9,bitmart,BitMart,24,46043.113954


,asset_id,coingecko_id,symbol,name,circulating_supply,total_supply,max_supply
0,1,bitcoin,BTC,Bitcoin,2.000281e+07,2.000281e+07,2.100000e+07
1,2,ethereum,ETH,Ethereum,1.206919e+08,1.206919e+08,NaN
2,3,tether,USDT,Tether,1.840323e+11,1.894993e+11,NaN
3,4,binancecoin,BNB,BNB,1.363579e+08,1.363579e+08,2.000000e+08
4,5,ripple,XRP,XRP,6.122783e+10,9.998570e+10,1.000000e+11
...,...,...,...,...,...,...,...
95,96,fetch-ai,FET,Artificial Superintelligence Alliance,2.260137e+09,2.714385e+09,2.714385e+09
96,97,virtual-protocol,VIRTUAL,Virtuals Protocol,6.563020e+08,1.000000e+09,1.000000e+09
97,98,true-usd,TUSD,TrueUSD,4.945151e+08,4.945151e+08,NaN
98,99,pancakeswap-token,CAKE,PancakeSwap,3.294786e+08,3.427484e+08,4.000000e+08


,date,asset_id,price_usd,market_cap_usd,global_volume_24h_usd,binance_volume_24h_usd
0,2026-03-13,1,73342.000000,1468078846334,5.181936e+10,2.591769e+09
1,2026-03-13,2,2279.590000,275379358516,3.396717e+10,1.279403e+09
2,2026-03-13,3,0.999985,184031727331,9.525152e+10,1.123857e+07
3,2026-03-13,4,672.520000,91790040092,1.614401e+09,1.539733e+08
4,2026-03-13,5,1.500000,91571365386,3.650243e+09,2.879646e+08
...,...,...,...,...,...,...
95,2026-03-13,96,0.234631,524098816,2.327987e+08,2.375050e+07
96,2026-03-13,97,0.774129,508538205,1.044817e+08,1.579813e+07
97,2026-03-13,98,0.999812,494318107,8.101426e+06,3.336478e+05
98,2026-03-13,99,1.490000,491644923,6.511604e+07,3.375700e+06


,asset_id,coingecko_id,symbol,name,market_cap_usd,global_volume_24h_usd,major_exchange_count,market_cap_rank,passes_volume_filter,passes_exchange_filter,passes_market_cap_filter,passes_all_filters
0,1,bitcoin,BTC,Bitcoin,1468078846334,5.181936e+10,17,1,True,True,True,True
1,2,ethereum,ETH,Ethereum,275379358516,3.396717e+10,18,2,True,True,True,True
2,3,tether,USDT,Tether,184031727331,9.525152e+10,20,3,True,True,True,True
3,4,binancecoin,BNB,BNB,91790040092,1.614401e+09,20,4,True,True,True,True
4,5,ripple,XRP,XRP,91571365386,3.650243e+09,20,5,True,True,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...
95,96,fetch-ai,FET,Artificial Superintelligence Alliance,524098816,2.327987e+08,20,96,True,True,True,True
96,97,virtual-protocol,VIRTUAL,Virtuals Protocol,508538205,1.044817e+08,17,97,True,True,True,True
97,98,true-usd,TUSD,TrueUSD,494318107,8.101426e+06,8,98,False,False,True,False
98,99,pancakeswap-token,CAKE,PancakeSwap,491644923,6.511604e+07,19,99,True,True,True,True


,asset_id,coingecko_id,symbol,name,market_cap_usd,global_volume_24h_usd,major_exchange_count,market_cap_rank,passes_volume_filter,passes_exchange_filter,passes_market_cap_filter,passes_all_filters
0,1,bitcoin,BTC,Bitcoin,1468078846334,5.181936e+10,17,1,True,True,True,True
1,2,ethereum,ETH,Ethereum,275379358516,3.396717e+10,18,2,True,True,True,True
2,3,tether,USDT,Tether,184031727331,9.525152e+10,20,3,True,True,True,True
3,4,binancecoin,BNB,BNB,91790040092,1.614401e+09,20,4,True,True,True,True
4,5,ripple,XRP,XRP,91571365386,3.650243e+09,20,5,True,True,True,True
5,6,usd-coin,USDC,USDC,79308339701,5.260844e+09,20,6,True,True,True,True
6,7,solana,SOL,Solana,53358252433,6.207191e+09,20,7,True,True,True,True
7,8,tron,TRX,TRON,28230429580,3.804510e+08,19,8,True,True,True,True
8,10,dogecoin,DOGE,Dogecoin,15380144495,2.311191e+09,20,10,True,True,True,True
9,13,cardano,ADA,Cardano,10471393453,9.529964e+08,20,13,True,True,True,True


In [15]:
# Debug Binance markets and raw method availability

print("ccxt exchange id:", binance.id)
print("number of markets:", len(binance.markets))

print("Has BTC/USDT:", "BTC/USDT" in binance.markets)
print("Has ETH/USDT:", "ETH/USDT" in binance.markets)
print("Has SOL/USDT:", "SOL/USDT" in binance.markets)

print("Has raw publicGetKlines method:", hasattr(binance, "publicGetKlines"))

ccxt exchange id: binance
number of markets: 4274
Has BTC/USDT: True
Has ETH/USDT: True
Has SOL/USDT: True
Has raw publicGetKlines method: True


In [16]:
# Debug single fetch for BTC

test_symbol = "BTC"
test_market_symbol = find_binance_usd_like_symbol(binance, test_symbol)

print("test_symbol:", test_symbol)
print("test_market_symbol:", test_market_symbol)

market = binance.market(test_market_symbol)
print("market id:", market["id"])

start_ts = binance.parse8601(f"{RUN_DATE}T00:00:00Z")
end_ts = start_ts + 24 * 60 * 60 * 1000

print("start_ts:", start_ts)
print("end_ts:", end_ts)

rows = binance.publicGetKlines(
    {
        "symbol": market["id"],
        "interval": "1d",
        "startTime": start_ts,
        "endTime": end_ts,
        "limit": 1,
    }
)

print("rows returned:", len(rows))
print("first row:", rows[0] if rows else None)

test_symbol: BTC
test_market_symbol: BTC/USDT
market id: BTCUSDT
start_ts: 1773360000000
end_ts: 1773446400000
rows returned: 1
first row: ['1773360000000', '70541.34000000', '73913.74000000', '70386.01000000', '70930.00000000', '35996.61678000', '1773446399999', '2591768526.29614710', '5312490', '18310.34260000', '1319530652.95863130', '0']
